In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/abhiaks19@gmail.com/databricks-data-engineering/consolidated_pipeline/Setup/utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
dbutils.widgets.text("catalog","fmcg","Catalog")
dbutils.widgets.text("data_source","customers","Data Source")


In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")
#print(catalog, data_source)

# to get all the data files from the s3 bucket/data_source(customers)
base_path=f's3://sportsbargmbh/{data_source}/*.csv'
print(base_path)


In [0]:
#creating datafrome from the above path
df=spark.read.format("csv").load(base_path)
display(df.limit(10))

In [0]:
#some formatting to the dataframe and adding metadata
df=(spark.read.format("csv")
    .option("header",True)
    .option("inferSchema",True)
    .load(base_path)
    .withColumn("read_timestamp",F.current_timestamp())
    .select("*","_metadata.file_name","_metadata.file_size")
)
#display(df.limit(10))
df.printSchema()

In [0]:
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed","true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")
#enable change data field will turn on CDF which help to allow tracking changes at row level

### Silver Layer Processing

In [0]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source}")
display(df_bronze.limit(10))


In [0]:
df_bronze.printSchema()

In [0]:
# Doing transformations as part of silver layer
# counting duplicates
df_duplicates = df_bronze.groupBy("customer_id").count().filter(F.col("count") > 1)
#display(df_duplicates)

# removing duplicates
df_silver = df_bronze.dropDuplicates(["customer_id"])

print('Rows before duplicate dropped: ', df_bronze.count())
print('Rows after duplicate dropped: ', df_silver.count())


In [0]:
#checking and removing leading spaces from customer_name
display(
    df_silver.filter(F.col("customer_name")!=F.trim(F.col("customer_name")))
)



In [0]:
# removing
df_silver=df_silver.withColumn("customer_name",F.trim(F.col("customer_name")))

In [0]:
# displaying distinct cities in order to clean up issues like fat fingering, duplicates etc.
display(
    df_silver.select("city").distinct().show())


In [0]:
# # typo dictionary
# city_typos = {
#     'Bengaluru': ['Bengaluruu', 'Bengaluruu', 'Bengalore'],
#     'Hyderabad': ['Hyderabadd', 'Hyderbad'],
#     'New Delhi': ['NewDelhi', 'NewDheli', 'NewDelhee']
# }

# typos → correct names
city_mapping = {
    'Bengaluruu': 'Bengaluru',
    'Bengalore': 'Bengaluru',

    'Hyderabadd': 'Hyderabad',
    'Hyderbad': 'Hyderabad',

    'NewDelhi': 'New Delhi',
    'NewDheli': 'New Delhi',
    'NewDelhee': 'New Delhi'
}


allowed = ["Bengaluru", "Hyderabad", "New Delhi"]

df_silver = (
    df_silver
    .replace(city_mapping, subset=["city"])
    .withColumn(
        "city",
        F.when(F.col("city").isNull(), None)
         .when(F.col("city").isin(allowed), F.col("city"))
         .otherwise(None)
    )
)

df_silver.select('city').distinct().show()

In [0]:
# displaying nulls

df_silver.filter(F.col("city").isNull()).show(truncate=False)


In [0]:
# from the data we know that these customer_names are also in different cities
null_customer_names=['Sprintx Nutrition','Zenathlete Foods','Primefuel Nutrition','Recovery Lane']
df_silver.filter(F.col("customer_name").isin(null_customer_names)).show(truncate=False)
# from above it is clear that only Bengaluru, Hyderabad and New Delhi are valid cities so for those customer names which already have locations in either 2 of three, it is common to guess to assume the third location, however for customers which only have less than 3 locations- we need to clarify those kind of things from business analyst


In [0]:

# Business Confirmation Note: City corrections confirmed by business team
customer_city_fix = {
    # Sprintx Nutrition
    789403: "New Delhi",

    # Zenathlete Foods
    789420: "Bengaluru",

    # Primefuel Nutrition
    789521: "Hyderabad",

    # Recovery Lane
    789603: "Hyderabad"
}

df_fix = spark.createDataFrame(
    [(k, v) for k, v in customer_city_fix.items()],
    ["customer_id", "fixed_city"]
)

display(df_fix)

In [0]:
df_silver=(
    df_silver.join(df_fix,"customer_id","left")
    .withColumn(
        "city", F.coalesce("city","fixed_city")
    )
    .drop("fixed_city")
)
display(df_silver)

In [0]:
# making ccustomer_casing consistent
df_silver.select('customer_name').distinct().show()


In [0]:
# initcap makes the first letter of each word uppercase
df_silver=df_silver.withColumn("customer_name",F.when(F.col("customer_name").isNull(),None)
                               .otherwise(F.initcap("customer_name")))

In [0]:
df_silver = df_silver.withColumn("customer_id", F.col("customer_id").cast("string"))
print(df_silver.printSchema())

In [0]:
display(df_silver)


In [0]:
# Now in order to merge this data from silver to child company data in gold, we need to do some more modifications as there are some columns which are not present in gold and names of some columns are not consistent
# 1-- Here in silver it is customer_id and in gold it is customer_code
# 2-- Here in silver it is customer_name but in gold it is customer
# 3-- In gold there are 3 extra columns market, platform and channel so we need to add those columns here in silver
# 4-- Here in silver there are 2 columns customer name and city however in gold there is only name and country, so its better to merger the customer name with city into one column 



In [0]:
df_silver = (
    df_silver
    # Build final customer column: "CustomerName-City" or "CustomerName-Unknown"
    .withColumn(
        "customer",
        F.concat_ws("-", "customer_name", F.coalesce(F.col("city"), F.lit("Unknown")))
    )
    
    # Static attributes aligned with parent data model
    .withColumn("market", F.lit("India"))
    .withColumn("platform", F.lit("Sports Bar"))
    .withColumn("channel", F.lit("Acquisition"))
)
display(df_silver.limit(10))

In [0]:
# writin to silver schema 
df_silver.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .option("mergeSchema", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

### GOLD Schema processing

In [0]:
df_silver = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source};")


# take req cols only
# "customer_id, customer_name, city, read_timestamp, file_name, file_size, customer, market, platform, channel"
df_gold = df_silver.select("customer_id", "customer_name", "city", "customer", "market", "platform", "channel")

In [0]:
df_gold.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

In [0]:
delta_table = DeltaTable.forName(spark,"fmcg.gold.dim_customers")
df_child_customers = spark.table("fmcg.gold.sb_dim_customers").select(F.col("customer_id").alias("customer_code"),"customer","market","platform","channel")

In [0]:
delta_table.alias("target").merge(
    source=df_child_customers.alias("source"),
    condition="target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()